# 01 — SAR dataset preparation (Colab)

Downloads, converts, validates and profiles one SAR detection dataset.

**Run this before any training.** The statistics it prints are what justify (or reject) individual components — in particular, the object-size histogram is the evidence required for the P2 small-object head.

Select **Runtime → Change runtime type → GPU** first.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'

## 1. Get the repository

Set `REPO` to your fork/clone URL. If you already have the repo in Drive, mount it instead.

In [ ]:
import os
REPO = 'https://github.com/officialarghya29/sarr-imaging.git'
WORK = '/content/sarr-imaging'

if not os.path.isdir(WORK):
    !git clone --depth 1 $REPO $WORK
%cd $WORK
!git log --oneline -1

In [ ]:
# Colab ships a recent torch/ultralytics-compatible stack; install only what is missing.
!pip -q install ultralytics pandas matplotlib pyyaml tqdm
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Fetch the dataset

Start with **SSDD** (pilot tier, ~1.2k images) — the whole ablation grid fits in a free session. Only scale up to HRSID / SARDet-100K once the pipeline is proven.

**Licensed download routes are listed in `docs/DATASETS.md`.** This repository never redistributes the data. Automated fetching is best-effort: dataset slugs on Kaggle/HuggingFace change, so verify and fall back to a manual upload if needed.

In [ ]:
DATASET = 'ssdd'          # ssdd | hrsid | sar_ship | sardet100k
RAW_DIR = '/content/raw'
os.makedirs(RAW_DIR, exist_ok=True)

# Option A: HuggingFace mirror (no credentials needed for public datasets)
try:
    !pip -q install huggingface_hub
    from huggingface_hub import snapshot_download
    from saryolo.data import get_dataset
    spec = get_dataset(DATASET)
    for repo in spec.hf_repos:
        try:
            path = snapshot_download(repo_id=repo, repo_type='dataset', local_dir=f'{RAW_DIR}/{DATASET}')
            print('downloaded from', repo, '->', path)
            break
        except Exception as exc:
            print('skipped', repo, ':', type(exc).__name__)
    else:
        print('No HuggingFace mirror worked for', DATASET)
except Exception as exc:
    print('HuggingFace route unavailable:', exc)

# Option B: Kaggle (needs an API token: upload kaggle.json to ~/.kaggle/)
# !pip -q install kagglehub
# import kagglehub, shutil
# from saryolo.data import get_dataset
# for slug in get_dataset(DATASET).kaggle_slugs:
#     try:
#         p = kagglehub.dataset_download(slug)
#         shutil.copytree(p, f'{RAW_DIR}/{DATASET}', dirs_exist_ok=True)
#         print('downloaded', slug); break
#     except Exception as e:
#         print('skipped', slug, e)

# Option C: upload the archive manually, then unzip here.
# from google.colab import files
# files.upload()
# !unzip -q -o archive.zip -d $RAW_DIR/$DATASET
print('\nRaw directory contents:')
!ls -R $RAW_DIR 2>/dev/null | head -30

## 3. Convert to YOLO format

Symlinks are used by default so a 100k-image benchmark does not duplicate tens of GB. Prefer the official split where the dataset ships one.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/prepare_dataset.py',
                '--dataset', DATASET, '--raw', f'{RAW_DIR}/{DATASET}',
                '--mode', 'symlink'], check=True)

## 4. Validate

Catches corrupt images, invalid class ids, out-of-range and zero-area boxes, orphan labels, and exact/near duplicates. Conversion problems are cheaper to find here than after a training run.

In [ ]:
from saryolo.data import get_dataset
names = ' '.join(get_dataset(DATASET).classes)
!python -m saryolo check-data --dataset datasets/processed/$DATASET --classes $names

## 5. Profile

Class distribution, objects per image, object-size histogram (COCO small/medium/large), aspect ratios, density heat map, and SAR difficulty proxies (local contrast, target/background ratio).

**Read the `size bins` line before enabling the P2 head.** If `small` does not dominate, the P2 head is not justified — it is the most expensive component in the model.

In [ ]:
!python -m saryolo stats --dataset datasets/processed/$DATASET --classes $names --name $DATASET

from IPython.display import Image, display
import glob
for f in sorted(glob.glob(f'dataset_statistics/{DATASET}*/size_distribution.png')) + \
         sorted(glob.glob(f'dataset_statistics/{DATASET}*/class_distribution.png')) + \
         sorted(glob.glob(f'dataset_statistics/{DATASET}*/object_size_hist.png')):
    display(Image(f))

## 6. Leakage check

Required for chip-based datasets (HRSID, SAR-Ship-Dataset) whose patches are cut from a few large scenes: a random chip split leaks near-identical patches across train/test and inflates mAP substantially.

In [ ]:
from pathlib import Path
from saryolo.data.splits import leakage_report

root = Path(f'datasets/processed/{DATASET}')
splits = {}
for split in ('train', 'val', 'test'):
    d = root / 'images' / split
    if d.is_dir():
        splits[split] = [p.stem for p in d.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')]

print({k: len(v) for k, v in splits.items()})
if len(splits) >= 2:
    report = leakage_report(splits, root / 'images' / 'train')
    print(report.summary())
else:
    print('Need at least two splits to check leakage.')

## 7. Persist to Drive

Colab VMs are ephemeral. Save the processed dataset and split lists so a later session can resume without re-downloading. (`datasets/raw` is gitignored; only the derived data and configs are worth keeping.)

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/saryolo && cp -r datasets configs/datasets dataset_statistics validation_reports /content/drive/MyDrive/saryolo/ 2>/dev/null
    print('saved to Drive')
print('\nNext: notebooks/02_train_and_ablate.ipynb')